# MedShield: Composable Clinical Triage Verifier & Safety Auditor Agent

**MedShield** is an autonomous clinical agent built on the **CROO Agent Protocol (CAP)**. It acts as an on-chain safety net and bias-auditing system for triage AI models. Other diagnosis or triage agents (Requesters) submit patient parameters, vitals, and their predicted triage level (Emergency Severity Index) to MedShield, which audits the triage for clinical danger flags and demographic bias, returning a structured Safety Audit Report settled on-chain via Base USDC.

### Target Hackathon Tracks:
1. **Data & Verification Agents** — Checks physiological consistency, vital anomalies, and patient safety flags.
2. **Research & Intelligence Agents** — Provides clinical reasoning explanations and audits systematic demographic undertriage bias.

---

## 1. Setup & Project Initialization
Run the following code block to create the project directory structure (`src/` and `tests/`) and write out the required source code files.

In [1]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('tests', exist_ok=True)
print('Project directory structure initialized successfully.')

Project directory structure initialized successfully.


### 1.1 Requirements File

In [2]:
%%writefile requirements.txt
croo-sdk>=0.1.0
pydantic>=2.0.0
websockets>=12.0
aiohttp>=3.9.0
httpx>=0.27.0


Writing requirements.txt


### 1.2 Configuration Module (`src/config.py`)
Handles toggling between local mock mode and live CROO network endpoints.

In [3]:
%%writefile src/config.py
import os

# ── CROO SDK Configurations ──────────────────────────────────────────────────
CROO_API_URL = os.getenv("CROO_API_URL", "https://api.croo.network")
CROO_WS_URL  = os.getenv("CROO_WS_URL", "wss://api.croo.network/ws")
CROO_SDK_KEY = os.getenv("CROO_SDK_KEY", "")

# ── Local Server Configurations (for testing) ────────────────────────────────
MOCK_API_PORT = int(os.getenv("MOCK_API_PORT", "8080"))
MOCK_WS_PORT  = int(os.getenv("MOCK_WS_PORT", "8081"))

# Fallbacks for mock URLs if running in local test mode
if os.getenv("MOCK_MODE", "false").lower() == "true":
    CROO_API_URL = f"http://localhost:{MOCK_API_PORT}"
    CROO_WS_URL  = f"ws://localhost:{MOCK_WS_PORT}/ws"


Writing src/config.py


### 1.3 Clinical Verifier Engine (`src/verifier.py`)
Contains the core clinical verification rules (Shock Index, Hypoxia, RR, Temp, GCS, Pain) and the demographic bias auditing logic.

In [4]:
%%writefile src/verifier.py
from pydantic import BaseModel, Field
from typing import List, Optional

# ── Input Schemas ─────────────────────────────────────────────────────────────

class DemographicsSchema(BaseModel):
    sex: str = Field(..., description="Patient sex: F, M, or Other")
    age: int = Field(..., description="Patient age in years")
    insurance_type: str = Field(..., description="Insurance: none, public, private, military, unknown")
    language: str = Field(..., description="Primary language: Finnish, English, Swedish, Russian, Arabic, Estonian, Somali, Other")
    arrival_mode: str = Field(..., description="Arrival: walk-in, ambulance, brought_by_family, police, transfer, helicopter")

class VitalsSchema(BaseModel):
    heart_rate: Optional[float] = None
    systolic_bp: Optional[float] = None
    diastolic_bp: Optional[float] = None
    respiratory_rate: Optional[float] = None
    spo2: Optional[float] = None
    temperature_c: Optional[float] = None
    gcs_total: Optional[int] = Field(default=15, description="Glasgow Coma Scale (3-15)")
    pain_score: Optional[float] = Field(default=0, description="Pain score (0-10)")

class AuditRequest(BaseModel):
    demographics: DemographicsSchema
    vitals: VitalsSchema
    chief_complaint_raw: str = Field(..., description="Free-text chief complaint")
    comorbidities: List[str] = Field(default_factory=list, description="List of hx_* flags (e.g. hx_heart_failure, hx_copd)")
    predicted_acuity: int = Field(..., description="The triage acuity (ESI 1-5) predicted by the primary AI or nurse")


# ── Output Schema ────────────────────────────────────────────────────────────

class AuditResponse(BaseModel):
    is_safe: bool = Field(..., description="True if no critical clinical warnings or undertriage bias detected")
    clinical_warnings: List[str] = Field(..., description="List of detected physiological danger flags")
    bias_undertriage_probability: float = Field(..., description="Empirical probability of systematic undertriage bias based on demographics")
    suggested_acuity: int = Field(..., description="The recommended ESI acuity level (1 = most critical, 5 = least urgent)")
    clinical_rationale: str = Field(..., description="Clinical reasoning explaining the safety warnings and suggested upgrade")


# ── Verifier Engine ──────────────────────────────────────────────────────────

def audit_triage(req: AuditRequest) -> AuditResponse:
    warnings = []
    
    # 1. Check vitals physiological danger signs
    hr = req.vitals.heart_rate
    sbp = req.vitals.systolic_bp
    rr = req.vitals.respiratory_rate
    spo2 = req.vitals.spo2
    temp = req.vitals.temperature_c
    gcs = req.vitals.gcs_total
    pain = req.vitals.pain_score
    
    # Shock Index (HR / SBP)
    if hr is not None and sbp is not None and sbp > 0:
        shock_index = hr / sbp
        if shock_index > 1.0:
            warnings.append(f"Haemodynamic Instability: Shock Index = {shock_index:.2f} (> 1.0)")
            
    # Hypoxia
    if spo2 is not None and spo2 < 94:
        warnings.append(f"Hypoxia: SpO2 = {spo2:.1f}% (< 94%)")
        
    # Tachypnea
    if rr is not None and rr > 20:
        warnings.append(f"Tachypnea: Respiratory Rate = {rr:.1f}/min (> 20/min)")
        
    # Fever or Hypothermia
    if temp is not None:
        if temp >= 38.0:
            warnings.append(f"Hyperthermia (Fever): Temperature = {temp:.1f}°C (>= 38.0°C)")
        elif temp < 36.0:
            warnings.append(f"Hypothermia: Temperature = {temp:.1f}°C (< 36.0°C)")
            
    # Altered consciousness
    if gcs is not None and gcs < 15:
        warnings.append(f"Altered Consciousness: GCS = {gcs} (< 15)")
        
    # Severe Pain
    if pain is not None and pain >= 8:
        warnings.append(f"Severe Pain: Pain Score = {pain:.1f} (>= 8)")

    # 2. Audit demographics for systematic undertriage bias
    # These probabilities reflect the empirical findings from our Triagegeist analysis:
    # Baseline undertriage risk is ~2.8%
    bias_score = 0.028
    
    # Language barrier risk
    non_primary_languages = ["Arabic", "Russian", "Somali", "Other"]
    if req.demographics.language in non_primary_languages:
        bias_score += 0.015  # elevated risk due to communication barrier
        
    # Insurance type risk
    high_risk_insurance = ["none", "unknown", "public"]
    if req.demographics.insurance_type in high_risk_insurance:
        bias_score += 0.010
        
    # Intersectional vulnerability: uninsured/underinsured women show highest historical rates
    if req.demographics.sex == "F" and req.demographics.insurance_type in ["none", "unknown"]:
        bias_score += 0.020
        
    # Age group risk
    if req.demographics.age >= 75:
        bias_score += 0.015  # geriatric atypical presentations
    elif req.demographics.age < 18:
        bias_score += 0.005  # pediatric vital signs discrepancy

    # 3. Calculate suggested acuity level (ESI 1-5)
    # ESI 1: Resuscitation (needs immediate life-saving intervention: GCS < 9, SpO2 < 85%, or severe shock)
    # ESI 2: High risk/Emergent (confused/lethargic GCS < 15, severe pain score >= 8, or multiple vital flags)
    # ESI 3: Urgent (needs multiple resources, vitals stable)
    # ESI 4: Less Urgent (needs one resource)
    # ESI 5: Non-Urgent (needs no resources)
    
    suggested = req.predicted_acuity
    
    # ESI 1 criteria
    if (gcs is not None and gcs <= 8) or (spo2 is not None and spo2 < 85):
        suggested = 1
    # ESI 2 criteria
    elif (gcs is not None and gcs < 15) or (pain is not None and pain >= 8) or len(warnings) >= 2 or (spo2 is not None and spo2 < 92):
        suggested = min(suggested, 2)
    # Upgrade to ESI 3 if there are any vital warnings
    elif len(warnings) > 0:
        suggested = min(suggested, 3)
        
    # 4. Check if the predicted acuity is safe
    # If the primaries predicted a level higher (meaning less critical) than the suggested level, it is unsafe.
    is_safe = True
    if req.predicted_acuity > suggested:
        is_safe = False
        
    # Create the rationale
    if not is_safe:
        warnings_str = ", ".join(warnings) if warnings else "demographic undertriage risk indicators"
        rationale = (
            f"Triage upgrade suggested: Level {req.predicted_acuity} -> Level {suggested}. "
            f"Audit flags: [{warnings_str}]. The primary predicted level is too low, "
            f"representing a potential undertriage error (calculated bias probability: {bias_score*100:.1f}%)."
        )
    else:
        rationale = "Triage acuity predictions are clinically consistent with vital signs and demographic baselines."
        
    return AuditResponse(
        is_safe=is_safe,
        clinical_warnings=warnings,
        bias_undertriage_probability=round(bias_score, 4),
        suggested_acuity=suggested,
        clinical_rationale=rationale
    )


Writing src/verifier.py


### 1.4 CROO WS Agent client (`src/agent.py`)
Establishes a WebSocket connection to the CAP network, listens for new negotiations, auto-accepts valid contracts, and submits safety deliverables.

In [5]:
%%writefile src/agent.py
import asyncio
import json
import logging
import os
import sys

# Ensure src is in python path
sys.path.insert(0, os.path.dirname(__file__))

from croo import (
    AgentClient, Config, EventType, Event,
    DeliverOrderRequest, DeliverableType
)
import config
from verifier import audit_triage, AuditRequest, AuditResponse

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("MedShieldAgent")


class MedShieldAgent:
    def __init__(self):
        # 1. Load config
        self.croo_config = Config(
            base_url=config.CROO_API_URL,
            ws_url=config.CROO_WS_URL
        )
        self.client = AgentClient(self.croo_config, config.CROO_SDK_KEY)
        self.stream = None
        self.running = False

    async def start(self):
        logger.info(f"Starting MedShield Agent...")
        logger.info(f"CROO API URL: {config.CROO_API_URL}")
        logger.info(f"CROO WS URL:  {config.CROO_WS_URL}")
        
        # 2. Connect WebSocket
        self.stream = await self.client.connect_websocket()
        self.running = True
        
        # 3. Register Event Listeners
        self.stream.on(EventType.NEGOTIATION_CREATED, self.on_negotiation_created)
        self.stream.on(EventType.ORDER_PAID, self.on_order_paid)
        self.stream.on(EventType.ORDER_COMPLETED, self.on_order_completed)
        self.stream.on(EventType.ORDER_REJECTED, self.on_order_rejected)
        
        logger.info("MedShield Agent is ONLINE and listening for events [OK]")
        
        # 4. Keep alive
        while self.running:
            await asyncio.sleep(1)

    async def stop(self):
        logger.info("Stopping MedShield Agent...")
        self.running = False
        if self.stream:
            await self.stream.close()
        await self.client.close()
        logger.info("MedShield Agent stopped.")

    # ── Event Handlers ────────────────────────────────────────────────────────

    def on_negotiation_created(self, e: Event):
        logger.info(f"Received negotiation request: {e.negotiation_id}")
        # Launch async handler
        asyncio.create_task(self._handle_negotiation(e.negotiation_id))

    def on_order_paid(self, e: Event):
        logger.info(f"Order payment confirmed on-chain: {e.order_id}")
        # Launch async handler
        asyncio.create_task(self._handle_delivery(e.order_id))

    def on_order_completed(self, e: Event):
        logger.info(f"Order completed successfully: {e.order_id}")

    def on_order_rejected(self, e: Event):
        logger.warning(f"Order rejected: {e.order_id}")

    # ── Async Core Operations ──────────────────────────────────────────────────

    async def _handle_negotiation(self, negotiation_id: str):
        try:
            negotiation = await self.client.get_negotiation(negotiation_id)
            logger.info(f"Negotiation Details: service={negotiation.service_id}, requester={negotiation.requester_agent_id}")
            
            # Auto-accept all incoming valid requests for our service
            logger.info(f"Accepting negotiation {negotiation_id}...")
            await self.client.accept_negotiation(negotiation_id)
            logger.info(f"Negotiation accepted successfully [OK]")
        except Exception as ex:
            logger.error(f"Error handling negotiation {negotiation_id}: {ex}", exc_info=True)

    async def _handle_delivery(self, order_id: str):
        try:
            logger.info(f"Processing order {order_id}...")
            
            # 1. Fetch Order and Negotiation to read the requirements (input payload)
            order = await self.client.get_order(order_id)
            negotiation = await self.client.get_negotiation(order.negotiation_id)
            
            logger.info(f"Raw requirements string: {negotiation.requirements}")
            
            # 2. Parse Requirements
            try:
                payload_dict = json.loads(negotiation.requirements)
                audit_req = AuditRequest(**payload_dict)
            except Exception as parse_err:
                logger.error(f"Failed to parse requirements payload: {parse_err}")
                # Reject order if input parameters are invalid/malformed
                await self.client.reject_order(order_id, f"Invalid requirements payload: {parse_err}")
                return
            
            # 3. Execute Triage Verification & Bias Auditing
            logger.info("Executing clinical verification & demographic bias audit...")
            audit_res = audit_triage(audit_req)
            logger.info(f"Audit completed: is_safe={audit_res.is_safe}, suggested_acuity={audit_res.suggested_acuity}")
            
            # 4. Prepare Delivery
            deliver_req = DeliverOrderRequest(
                deliverable_type=DeliverableType.SCHEMA,
                deliverable_schema=audit_res.model_dump_json()
            )
            
            # 5. Submit Delivery to CROO
            logger.info(f"Submitting delivery for order {order_id}...")
            await self.client.deliver_order(order_id, deliver_req)
            logger.info(f"Order {order_id} successfully delivered [OK]")
            
        except Exception as ex:
            logger.error(f"Error executing delivery for order {order_id}: {ex}", exc_info=True)
            try:
                await self.client.reject_order(order_id, f"Internal execution error: {ex}")
            except Exception as reject_ex:
                logger.error(f"Failed to reject order: {reject_ex}")


# ── Executable Entrypoint ────────────────────────────────────────────────────

if __name__ == "__main__":
    agent = MedShieldAgent()
    loop = asyncio.get_event_loop()
    try:
        loop.run_until_complete(agent.start())
    except KeyboardInterrupt:
        logger.info("KeyboardInterrupt received, shutting down...")
        loop.run_until_complete(agent.stop())
    finally:
        loop.close()


Writing src/agent.py


## 2. Testing & Local Simulation
To verify the complete Agent-to-Agent (A2A) protocol negotiation, payment, and delivery lifecycle, we write a local HTTP/WS mock server and an integration test runner.

### 2.1 Mock CROO Protocol Server (`tests/mock_server.py`)

In [6]:
%%writefile tests/mock_server.py
import asyncio
import json
import logging
import sys
from aiohttp import web, WSMsgType

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] MockServer: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("MockServer")

# In-memory database of negotiations, orders, and deliveries
db = {
    "negotiations": {},
    "orders": {},
    "deliveries": {}
}

# Active WebSocket connections
active_connections = []

async def broadcast_ws(event_payload: dict):
    """Send an event broadcast to all connected WebSocket clients."""
    message = json.dumps(event_payload)
    logger.info(f"WS Broadcast: {event_payload.get('data', {}).get('type')} -> {len(active_connections)} clients")
    for ws in active_connections:
        try:
            await ws.send_str(message)
        except Exception as e:
            logger.error(f"Failed to send to client: {e}")

# ── HTTP API Routes ───────────────────────────────────────────────────────────

async def handle_negotiate_order(request):
    """POST /backend/v1/orders/negotiate"""
    try:
        req_body = await request.json()
        logger.info(f"HTTP: POST /orders/negotiate details={req_body}")
        
        negotiation_id = f"neg_{len(db['negotiations']) + 1}"
        negotiation = {
            "negotiationId": negotiation_id,
            "serviceId": req_body.get("serviceId", "medshield-safety-audit"),
            "requesterAgentId": req_body.get("requesterAgentId", "requester_agent_mock"),
            "providerAgentId": "medshield_provider_mock",
            "requirements": req_body.get("requirements", ""),
            "status": "pending",
            "rejectReason": "",
            "metadata": req_body.get("metadata", ""),
            "expiresAt": "2026-06-21T00:00:00Z",
            "createdTime": "2026-06-20T17:18:00Z",
            "updatedTime": "2026-06-20T17:18:00Z",
            "fundAmount": req_body.get("fundAmount", ""),
            "fundToken": req_body.get("fundToken", ""),
            "providerFundAddress": ""
        }
        
        db["negotiations"][negotiation_id] = negotiation
        
        # Broadcast Event after a minor delay to ensure WebSocket is subscribed
        asyncio.create_task(trigger_negotiation_event(negotiation_id))
        
        return web.json_response({"negotiation": negotiation})
    except Exception as e:
        logger.error(f"Error in handle_negotiate_order: {e}")
        return web.json_response({"error": str(e)}, status=500)

async def trigger_negotiation_event(negotiation_id: str):
    await asyncio.sleep(0.5)
    neg = db["negotiations"].get(negotiation_id)
    if neg:
        event = {
            "data": {
                "type": "order_negotiation_created",
                "negotiation_id": negotiation_id,
                "service_id": neg["serviceId"],
                "requester_agent_id": neg["requesterAgentId"],
                "provider_agent_id": neg["providerAgentId"],
                "status": "pending"
            }
        }
        await broadcast_ws(event)

async def handle_accept_negotiation(request):
    """POST /backend/v1/orders/negotiate/{id}/accept"""
    try:
        negotiation_id = request.match_info['id']
        logger.info(f"HTTP: POST /orders/negotiate/{negotiation_id}/accept")
        
        if negotiation_id not in db["negotiations"]:
            return web.json_response({"reason": "Negotiation not found"}, status=404)
            
        neg = db["negotiations"][negotiation_id]
        neg["status"] = "accepted"
        
        order_id = f"order_{negotiation_id}"
        order = {
            "orderId": order_id,
            "negotiationId": negotiation_id,
            "chainOrderId": f"chain_{order_id}",
            "serviceId": neg["serviceId"],
            "requesterAgentId": neg["requesterAgentId"],
            "providerAgentId": neg["providerAgentId"],
            "buyerUserId": "buyer_mock",
            "requesterWalletAddress": "", # Bypasses balance check
            "providerWalletAddress": "",
            "price": "0",  # Bypasses balance check
            "paymentToken": "", # Bypasses balance check
            "deliveryWindow": 3600,
            "status": "created",
            "createdTime": "2026-06-20T17:18:00Z",
            "updatedTime": "2026-06-20T17:18:00Z",
            "createdAt": "2026-06-20T17:18:00Z"
        }
        
        db["orders"][order_id] = order
        return web.json_response({"negotiation": neg, "order": order})
    except Exception as e:
        logger.error(f"Error in handle_accept_negotiation: {e}")
        return web.json_response({"error": str(e)}, status=500)

async def handle_get_negotiation(request):
    """GET /backend/v1/orders/negotiate/{id}"""
    negotiation_id = request.match_info['id']
    logger.info(f"HTTP: GET /orders/negotiate/{negotiation_id}")
    neg = db["negotiations"].get(negotiation_id)
    if not neg:
        return web.json_response({"reason": "Negotiation not found"}, status=404)
    return web.json_response({"negotiation": neg})

async def handle_get_order(request):
    """GET /backend/v1/orders/{id}"""
    order_id = request.match_info['id']
    logger.info(f"HTTP: GET /orders/{order_id}")
    order = db["orders"].get(order_id)
    if not order:
        return web.json_response({"reason": "Order not found"}, status=404)
    return web.json_response({"order": order})

async def handle_pay_order(request):
    """POST /backend/v1/orders/{id}/pay"""
    try:
        order_id = request.match_info['id']
        logger.info(f"HTTP: POST /orders/{order_id}/pay")
        
        if order_id not in db["orders"]:
            return web.json_response({"reason": "Order not found"}, status=404)
            
        order = db["orders"][order_id]
        order["status"] = "paid"
        
        # Broadcast payment event after response
        asyncio.create_task(trigger_paid_event(order_id))
        
        return web.json_response({"order": order, "txHash": "0xmockpaytxhash"})
    except Exception as e:
        logger.error(f"Error in handle_pay_order: {e}")
        return web.json_response({"error": str(e)}, status=500)

async def trigger_paid_event(order_id: str):
    await asyncio.sleep(0.5)
    order = db["orders"].get(order_id)
    if order:
        event = {
            "data": {
                "type": "order_paid",
                "order_id": order_id,
                "negotiation_id": order["negotiationId"],
                "service_id": order["serviceId"],
                "status": "paid"
            }
        }
        await broadcast_ws(event)

async def handle_deliver_order(request):
    """POST /backend/v1/orders/{id}/deliver"""
    try:
        order_id = request.match_info['id']
        req_body = await request.json()
        logger.info(f"HTTP: POST /orders/{order_id}/deliver details={req_body}")
        
        if order_id not in db["orders"]:
            return web.json_response({"reason": "Order not found"}, status=404)
            
        order = db["orders"][order_id]
        order["status"] = "completed"
        
        delivery_id = f"del_{order_id}"
        delivery = {
            "deliveryId": delivery_id,
            "orderId": order_id,
            "providerAgentId": order["providerAgentId"],
            "deliverableType": req_body.get("deliverableType", "schema"),
            "deliverableSchema": req_body.get("deliverableSchema", ""),
            "deliverableText": req_body.get("deliverableText", ""),
            "status": "submitted",
            "submittedAt": "2026-06-20T17:18:00Z"
        }
        
        db["deliveries"][order_id] = delivery
        
        # Broadcast completion event
        asyncio.create_task(trigger_completed_event(order_id))
        
        return web.json_response({
            "order": order,
            "delivery": delivery,
            "txHash": "0xmockdelivertxhash"
        })
    except Exception as e:
        logger.error(f"Error in handle_deliver_order: {e}")
        return web.json_response({"error": str(e)}, status=500)

async def trigger_completed_event(order_id: str):
    await asyncio.sleep(0.5)
    order = db["orders"].get(order_id)
    if order:
        event = {
            "data": {
                "type": "order_completed",
                "order_id": order_id,
                "negotiation_id": order["negotiationId"],
                "service_id": order["serviceId"],
                "status": "completed"
            }
        }
        await broadcast_ws(event)

async def handle_reject_order(request):
    """POST /backend/v1/orders/{id}/reject"""
    try:
        order_id = request.match_info['id']
        req_body = await request.json()
        logger.info(f"HTTP: POST /orders/{order_id}/reject reason={req_body.get('reason')}")
        
        if order_id not in db["orders"]:
            return web.json_response({"reason": "Order not found"}, status=404)
            
        order = db["orders"][order_id]
        order["status"] = "rejected"
        
        # Broadcast reject event
        asyncio.create_task(trigger_rejected_event(order_id, req_body.get("reason", "")))
        
        return web.json_response({"order": order})
    except Exception as e:
        logger.error(f"Error in handle_reject_order: {e}")
        return web.json_response({"error": str(e)}, status=500)

async def trigger_rejected_event(order_id: str, reason: str):
    await asyncio.sleep(0.5)
    order = db["orders"].get(order_id)
    if order:
        event = {
            "data": {
                "type": "order_rejected",
                "order_id": order_id,
                "negotiation_id": order["negotiationId"],
                "service_id": order["serviceId"],
                "reason": reason,
                "status": "rejected"
            }
        }
        await broadcast_ws(event)

async def handle_get_delivery(request):
    """GET /backend/v1/orders/{id}/delivery"""
    order_id = request.match_info['id']
    logger.info(f"HTTP: GET /orders/{order_id}/delivery")
    delivery = db["deliveries"].get(order_id)
    if not delivery:
        return web.json_response({"reason": "Delivery not found"}, status=404)
    return web.json_response({"delivery": delivery})


# ── WebSocket Handler ─────────────────────────────────────────────────────────

async def handle_ws(request):
    """WS /ws"""
    ws = web.WebSocketResponse()
    await ws.prepare(request)
    
    sdk_key = request.query.get("key", "unknown")
    logger.info(f"WS: Client connected key={sdk_key}")
    active_connections.append(ws)
    
    try:
        async for msg in ws:
            if msg.type == WSMsgType.TEXT:
                # Do nothing, agent is typically receiver
                pass
            elif msg.type == WSMsgType.ERROR:
                logger.error(f"WS connection closed with exception {ws.exception()}")
    finally:
        active_connections.remove(ws)
        logger.info(f"WS: Client disconnected key={sdk_key}")
        
    return ws


# ── Main Runners ──────────────────────────────────────────────────────────────

async def start_mock_server():
    # 1. Setup API App
    api_app = web.Application()
    api_app.add_routes([
        web.post('/backend/v1/orders/negotiate', handle_negotiate_order),
        web.post('/backend/v1/orders/negotiate/{id}/accept', handle_accept_negotiation),
        web.get('/backend/v1/orders/negotiate/{id}', handle_get_negotiation),
        web.get('/backend/v1/orders/{id}', handle_get_order),
        web.post('/backend/v1/orders/{id}/pay', handle_pay_order),
        web.post('/backend/v1/orders/{id}/deliver', handle_deliver_order),
        web.post('/backend/v1/orders/{id}/reject', handle_reject_order),
        web.get('/backend/v1/orders/{id}/delivery', handle_get_delivery),
    ])
    
    api_runner = web.AppRunner(api_app)
    await api_runner.setup()
    api_site = web.TCPSite(api_runner, 'localhost', 8080)
    await api_site.start()
    logger.info("HTTP API server listening on http://localhost:8080")
    
    # 2. Setup WebSocket App
    ws_app = web.Application()
    ws_app.add_routes([
        web.get('/ws', handle_ws)
    ])
    
    ws_runner = web.AppRunner(ws_app)
    await ws_runner.setup()
    ws_site = web.TCPSite(ws_runner, 'localhost', 8081)
    await ws_site.start()
    logger.info("WebSocket server listening on ws://localhost:8081/ws")
    
    # Keep alive
    try:
        while True:
            await asyncio.sleep(3600)
    except asyncio.CancelledError:
        logger.info("Shutting down mock servers...")
    finally:
        await api_runner.cleanup()
        await ws_runner.cleanup()

if __name__ == "__main__":
    try:
        asyncio.run(start_mock_server())
    except KeyboardInterrupt:
        logger.info("Mock server stopped by user.")


Writing tests/mock_server.py


### 2.2 Integration Test Suite (`tests/run_test.py`)

In [7]:
%%writefile tests/run_test.py
import asyncio
import json
import logging
import os
import subprocess
import sys
import time

# Ensure src is in python path
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "src")))

# Force mock mode and provider keys
os.environ["MOCK_MODE"] = "true"
os.environ["CROO_SDK_KEY"] = "mock-provider-sdk-key"

from agent import MedShieldAgent
import config
from croo import AgentClient, Config, NegotiateOrderRequest

# Configure test logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] TestRunner: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("TestRunner")

# Mock payloads representing the requester's clinical cases
CASE_A_SAFE = {
    "demographics": {
        "sex": "M",
        "age": 35,
        "insurance_type": "private",
        "language": "English",
        "arrival_mode": "walk-in"
    },
    "vitals": {
        "heart_rate": 72.0,
        "systolic_bp": 120.0,
        "diastolic_bp": 80.0,
        "respiratory_rate": 16.0,
        "spo2": 98.0,
        "temperature_c": 36.8,
        "gcs_total": 15,
        "pain_score": 2.0
    },
    "chief_complaint_raw": "Mild ankle sprain after slipping on ice.",
    "comorbidities": [],
    "predicted_acuity": 4
}

CASE_B_UNSAFE = {
    "demographics": {
        "sex": "F",
        "age": 78,
        "insurance_type": "none",
        "language": "Arabic",
        "arrival_mode": "ambulance"
    },
    "vitals": {
        "heart_rate": 110.0,
        "systolic_bp": 90.0,
        "diastolic_bp": 60.0,
        "respiratory_rate": 24.0,
        "spo2": 84.0,
        "temperature_c": 38.5,
        "gcs_total": 14,
        "pain_score": 6.0
    },
    "chief_complaint_raw": "Shortness of breath and fever, confusion noted by family.",
    "comorbidities": ["hx_copd", "hx_heart_failure"],
    "predicted_acuity": 4
}

async def run_scenario(requester_client: AgentClient, case_name: str, payload: dict):
    logger.info(f"\n==========================================")
    logger.info(f"Running Scenario: {case_name}")
    logger.info(f"==========================================\n")
    
    # 1. Requester submits a negotiation request
    logger.info("Requester: Submitting order negotiation request...")
    req = NegotiateOrderRequest(
        service_id="medshield-safety-audit",
        requirements=json.dumps(payload)
    )
    
    negotiation = await requester_client.negotiate_order(req)
    neg_id = negotiation.negotiation_id
    logger.info(f"Requester: Negotiation created with ID: {neg_id}")
    
    # 2. Wait for provider to accept (which updates negotiation status to accepted & creates order)
    logger.info("Requester: Waiting for provider to accept negotiation...")
    order_id = f"order_{neg_id}"
    
    for attempt in range(10):
        try:
            order = await requester_client.get_order(order_id)
            if order.status == "created":
                logger.info(f"Requester: Order created successfully with ID: {order_id}")
                break
        except Exception:
            pass
        await asyncio.sleep(0.5)
    else:
        raise RuntimeError("Negotiation acceptance timed out. Provider agent failed to auto-accept.")
        
    # 3. Requester pays for the order
    logger.info(f"Requester: Paying for order {order_id}...")
    pay_res = await requester_client.pay_order(order_id)
    logger.info(f"Requester: Payment successful, Tx Hash: {pay_res.tx_hash}")
    
    # 4. Wait for the delivery to be completed by the provider agent
    logger.info("Requester: Waiting for audit delivery...")
    delivery = None
    for attempt in range(10):
        try:
            order = await requester_client.get_order(order_id)
            if order.status == "completed":
                delivery = await requester_client.get_delivery(order_id)
                logger.info("Requester: Audit report delivered! [OK]")
                break
        except Exception as e:
            logger.debug(f"Waiting for delivery error: {e}")
        await asyncio.sleep(0.5)
    else:
        raise RuntimeError("Audit delivery timed out.")
        
    # 5. Verify the Audit Response
    logger.info("\n--- AUDIT RESULTS RECEIVED ---")
    logger.info(f"Deliverable Type: {delivery.deliverable_type}")
    logger.info(f"Delivery Status:  {delivery.status}")
    
    audit_data = json.loads(delivery.deliverable_schema)
    logger.info(f"Is Safe:          {audit_data.get('is_safe')}")
    logger.info(f"Suggested Acuity: {audit_data.get('suggested_acuity')}")
    logger.info(f"Warnings:         {audit_data.get('clinical_warnings')}")
    logger.info(f"Bias Undertriage Probability: {audit_data.get('bias_undertriage_probability') * 100:.2f}%")
    logger.info(f"Rationale:        {audit_data.get('clinical_rationale')}")
    logger.info("-------------------------------\n")
    
    return audit_data

async def main():
    # 1. Start mock server as a background subprocess
    logger.info("Starting local mock server...")
    server_proc = subprocess.Popen(
        [sys.executable, "tests/mock_server.py"],
        stdout=None,
        stderr=None
    )
    
    # Give the server a moment to bind ports
    time.sleep(2)
    
    # Check if mock server started successfully
    if server_proc.poll() is not None:
        stdout, stderr = server_proc.communicate()
        logger.error(f"Mock server failed to start. Stdout: {stdout}, Stderr: {stderr}")
        sys.exit(1)
        
    # 2. Start MedShield Provider Agent
    logger.info("Starting MedShield Provider Agent...")
    provider_agent = MedShieldAgent()
    provider_task = asyncio.create_task(provider_agent.start())
    
    # 3. Start Requester Client
    requester_config = Config(
        base_url=config.CROO_API_URL,
        ws_url=config.CROO_WS_URL
    )
    requester_client = AgentClient(requester_config, "mock-requester-sdk-key")
    
    try:
        # Run Scenario A (Safe triage)
        res_a = await run_scenario(requester_client, "Case A: Safe Mild Case", CASE_A_SAFE)
        assert res_a["is_safe"] is True
        assert res_a["suggested_acuity"] == 4
        assert len(res_a["clinical_warnings"]) == 0
        
        # Run Scenario B (Unsafe severe case needing upgrade)
        res_b = await run_scenario(requester_client, "Case B: Critical Undertriage Case", CASE_B_UNSAFE)
        assert res_b["is_safe"] is False
        assert res_b["suggested_acuity"] == 1
        assert len(res_b["clinical_warnings"]) > 0
        
        logger.info("All end-to-end clinical audit test cases PASSED successfully! [SUCCESS]")
        
    except Exception as e:
        logger.error(f"Test suite failed: {e}", exc_info=True)
        sys.exit(1)
    finally:
        # Cleanup
        logger.info("Stopping provider agent and mock server...")
        await provider_agent.stop()
        provider_task.cancel()
        
        server_proc.terminate()
        server_proc.wait()
        logger.info("Mock server stopped. Cleanup finished.")

if __name__ == "__main__":
    asyncio.run(main())


Writing tests/run_test.py


## 3. Execution & Verification Run
Now we install the required packages and run the end-to-end integration tests live in the notebook!

In [8]:
!pip install -r requirements.txt

In [9]:
!python tests/run_test.py

2026-06-20 14:46:06,299 [INFO] TestRunner: Starting local mock server...
2026-06-20 14:46:06,708 [INFO] MockServer: HTTP API server listening on http://localhost:8080
2026-06-20 14:46:06,709 [INFO] MockServer: WebSocket server listening on ws://localhost:8081/ws
2026-06-20 14:46:08,300 [INFO] TestRunner: Starting MedShield Provider Agent...
2026-06-20 14:46:08,771 [INFO] TestRunner: 
2026-06-20 14:46:08,771 [INFO] TestRunner: Running Scenario: Case A: Safe Mild Case
2026-06-20 14:46:08,771 [INFO] TestRunner: ==========================================

2026-06-20 14:46:08,771 [INFO] TestRunner: Requester: Submitting order negotiation request...
2026-06-20 14:46:08,810 [INFO] MedShieldAgent: Starting MedShield Agent...
2026-06-20 14:46:08,810 [INFO] MedShieldAgent: CROO API URL: http://localhost:8080
2026-06-20 14:46:08,810 [INFO] MedShieldAgent: CROO WS URL:  ws://localhost:8081/ws
2026-06-20 14:46:08,811 [INFO] croo: websocket connecting: ws://localhost:8081/ws?key=mock-provider-sdk-ke

## 4. Submission & Production Registration Guidelines
To deploy MedShield to the production CROO network:
1. Secure an SDK key from the [CROO Agent Dashboard](https://agent.croo.network/).
2. Register the service ID `medshield-safety-audit` defining input schemas matching `AuditRequest` and output schemas matching `AuditResponse`.
3. Set production environment variables:
   ```bash
   export CROO_SDK_KEY="your-production-sdk-key"
   export CROO_API_URL="https://api.croo.network"
   export CROO_WS_URL="wss://api.croo.network/ws"
   python src/agent.py
   ```